# OLS - finding significant treatments 

This notebook uses the features data from the folder data/processed and creates additional features.  

In [10]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from c08_farming_exit import config
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "features_data.csv")

In [12]:
geo_cols = ['district', 'enumeration_area', 'region'] #these columns are missing for Botwana and Namibia
identification_cols = ['personal_id',  'interview_key', 'members_id']
df = df.drop(columns = geo_cols + identification_cols)

In [13]:
Y = [
     # 'empl_category', #categorial
     # 'farm_empl_last_12_months',
     # 'aspiration_continue_farming',
     # 'child_aspiration_continue_farming',

    # working hours  - absolute
     'farm_empl_hours_per_year',
     'off_farm_hours_per_year',

    # working hours  - relative
     'on_farm_hours_annual_share',
     'off_farm_hours_annual_share',

    # off farm income  - absolute
     'off_farm_income_per_year'
     ]

In [14]:
T = [
        'land_size_cropland_acres',
        'land_size_fallow_acres',
        'land_size_agroforestry_forestry_acres',
        'land_size_pasture_acres',
        'land_number_of_plots',

        'livestock_contract',
        'crop_contract',
        'membership_farmers_group',
        'membership_agricultural_cooperative',

        'livestock_market_distance_in_km',
        'road_type',
        'road_condition',
        'road_distance_in_minutes',
        'market_output_distance_in_km',
        'market_input_distance_in_km',

        'subsidy',

        'internet_access',
        'mobile_money_access',

        'shock_type_affected_last_12_months_crop_failure',
        'shock_type_affected_last_12_months_drought',
        'shock_type_affected_last_12_months_floods',
        'shock_type_affected_last_12_months_livestock_loss',
        'shock_type_affected_last_12_months_price_shock',
        'shock_coping_strategy_food_reduction',

        'agriculture_loan_last_5_years',
]

In [15]:
X = [
        #geographics
        'country', 

        #socio-demographics
        'female',
        'relation_to_head',
        'age',
        'years_of_schooling',
        'ethnic_group',
        'religion',
        'answered_for_self', 
    
        #land ownership
        'land_size_residential_acres',
        'land_size_lodge_camp_acres',
        'land_size_cropland_acres',
        'land_size_fallow_acres',
        'land_size_agroforestry_forestry_acres',
        'land_size_pasture_acres',
        'land_cropland_ownership_status',
        'land_residential_ownership_status',
        'grazing_land_ownership_status',
        'grazing_land_number_hh_sharing',
        'grazing_land_permit',
        'grazing_land_permit_price',
        'grazing_land_use_duration_in_years',
        'land_number_of_plots',

        #contracts/membership
        'livestock_contract',
        'crop_contract',
        'crop_contract_crop_type',
        'membership_farmers_group',
        'membership_agricultural_cooperative',

        #market distance and road quality
        'livestock_market_distance_in_km',
        'road_type',
        'road_condition',
        'road_distance_in_minutes',
        'market_output_distance_in_km',
        'market_input_distance_in_km',
        'market_type',

        #subsidies
        'subsidy',
        'subsidy_type_seeds',
        'subsidy_type_fertilizer',
        'subsidy_type_agro_chemicals',
        'subsidy_type_interest_free_loan',
        'subsidy_supplier_government',
        'subsidy_supplier_ngos',
        'subsidy_supplier_company',

        #assets
        'asset_value',
        'asset_diversity',
        'house_room_number',
        'house_roof_material',
        'house_wall_material',
        'house_floor_material',
        'house_water_source',
        'house_toilet_type',
        'house_energy_source',
        'house_energy_source_for_cooking',
        'house_energy_source_for_lighting',

        #mobile money and internet
        'internet_access',
        'internet_access_at_home',
        'mobile_money_access',
        
        #emotions
        'worry_about_job_loss_or_economic_livelihood',
        'life_satisfaction',
        'hh_optimism_index',
        'hh_sentiment_index',

        #add ons
        'sufficent_food_number_of_month',
        'hh_members_count',
        'beans_allocated_to_empl_occupation',
        'grazing_land_challenges_index',

        #other income/remittances
        'other_income_amount_annual',
        'remittance_amount_sent_last_12_months',

        #shock/coping
        'shock_type_affected_last_12_months_crop_failure',
        'shock_type_affected_last_12_months_drought',
        'shock_type_affected_last_12_months_floods',
        'shock_type_affected_last_12_months_illness_death',
        'shock_type_affected_last_12_months_livestock_loss',
        'shock_type_affected_last_12_months_other',
        'shock_type_affected_last_12_months_price_shock',
        'shock_coping_strategy_relatives_friends',
        'shock_coping_strategy_government',
        'shock_coping_strategy_food_reduction',
        'shock_coping_strategy_changed_cropping_practices',
        'shock_coping_strategy_more_employment',
        'shock_coping_strategy_hh_member_migration',
        'shock_coping_strategy_savings',
        'shock_coping_strategy_insurance',
        'shock_coping_strategy_credit',
        'shock_coping_strategy_sold_hh_assets',
        'shock_coping_strategy_sold_livestock',
        'shock_coping_strategy_migration',
        'shock_future_likelihood_change_income_source',

        #migration
        'migrant_last_12_months',
        'current_migrant',
        'migration_intention_next_12_months',
        
        #financing 
        'land_used_as_collateral',
        'agriculture_loan_last_5_years',
        'agriculture_loan_amount',
        'agriculture_loan_lender_bank',
        'agriculture_loan_lender_credit_union',
        'agriculture_loan_lender_private_lender',
        'agriculture_loan_lender_government',
        'agriculture_loan_lender_ngos',
 ]

In [16]:
full_df = df[Y + X]

In [17]:
for i in Y:
    #keep the target of interest and the features
    d = full_df[[i] + X].copy()      

    #cannot train on an emtry label
    d = d.dropna(subset=[i])        

    #drop all rows that have missing rates higher 60%
    keep = d.columns[d.isna().mean() <= 0.60]
    d = d[keep]

    # 1. Select numeric columns (float64 and int64) and categorical columns
    num_cols = d.select_dtypes(include=['float64', 'int64']).columns
    obj_cols = d.select_dtypes(include=['object']).columns

    # 2. Outlier treatment: clip values outside the 1%/99% quantiles, ignoring NaNs
    for col in num_cols:
        lower, upper = d[col].quantile(0.01), d[col].quantile(0.99)
        d[col] = d[col].clip(lower=lower, upper=upper)

    # 3. Fill numeric NaNs with country median
    d[num_cols] = (
        d[num_cols]
        .fillna(d.groupby("country")[num_cols].transform("median"))
        .fillna(d[num_cols].median())
    )

    # 4. Fill object NaNs/Nones with "missing"
    d[obj_cols] = d[obj_cols].fillna("missing")

    # Run OLS
    new_X = d.drop(columns=[i])
    new_X = pd.get_dummies(new_X, drop_first=True, dtype=float, dummy_na=False)

    X_const = sm.add_constant(new_X)
    model = sm.OLS(d[i], X_const).fit()

    print("---------------------------------------------------------------------")
    print("----------------------------**********-------------------------------")
    print("---------------------------------------------------------------------")
    print(f"Here comes the summary for {i}")
    # print(model.summary())

    results_table = pd.DataFrame({
    'coefficient': model.params,
    'p_value': model.pvalues,
    # 'std_err': model.bse,
    # 'conf_low': model.conf_int()[0],
    # 'conf_high': model.conf_int()[1]
    }).round(3)
    results_table_T = (
        results_table.loc[results_table.index.intersection(T)]
        .sort_values('p_value')
    )
    print(results_table_T)

    r_squared = model.rsquared
    adj_r_squared = model.rsquared_adj
    print(f"r_squared: {r_squared}, adj_r_squared: {adj_r_squared}")

    

---------------------------------------------------------------------
----------------------------**********-------------------------------
---------------------------------------------------------------------
Here comes the summary for farm_empl_hours_per_year
                                                   coefficient  p_value
land_size_pasture_acres                                 25.288    0.000
land_number_of_plots                                   157.731    0.000
shock_type_affected_last_12_months_price_shock        -472.368    0.000
land_size_cropland_acres                                27.739    0.001
subsidy                                                522.090    0.001
mobile_money_access                                    276.985    0.003
shock_type_affected_last_12_months_floods              548.959    0.004
shock_coping_strategy_food_reduction                   320.709    0.004
internet_access                                       -324.317    0.011
agriculture_loan_l